In [1]:

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
pd.options.mode.chained_assignment = None

import pyfade

In [ ]:
PUMP = 'A-24 1'
SOURCE = 'C:/Users/Heitor/Documents/PosDoc/code/EPIC/Dados/merged_data.csv'
EXAMPLE_SIGNAL = 'ESP discharge pressure'
# EXAMPLE_SIGNAL = None

In [3]:
IGNORE_START = True

SUB_SIZE = 1*24

USE_CUDA = True
LEFT_ONLY=True
SKIP_START=SUB_SIZE*5
QUANTILE = 0.75

MP_PARAMS = {
    'use_cuda': USE_CUDA,
    'left_only': LEFT_ONLY,
    'skip_start': SKIP_START,
    'quantile_threshold': QUANTILE,
    # 'exclusion_zone_ratio':
}


LEVELS = 5
DIMENSION = 4
WAVELET = 'db2'

WAVELET_PARAMS = {
    'wavelet_total_level': LEVELS,
    'wavelet_aggregate_level': DIMENSION,
    'wavelet_type': WAVELET
}

# 1 - Reading Signal

In [ ]:
# Load well data
if 'well_data' not in locals():
    data_full = pyfade.load_data(SOURCE)
    well_data = pyfade.get_well_data(data_full, PUMP, only_numerical=True, remove_ints=True, drop_na=True, drop_features = True)

shutdowns_all = pyfade.get_shutdowns(well_data)

In [ ]:
if EXAMPLE_SIGNAL is not None:
    sub_data = well_data[EXAMPLE_SIGNAL]
    shutdowns = [shutdowns_all[well_data.columns.get_loc(EXAMPLE_SIGNAL)]]
else:
    sub_data = well_data.copy()
    shutdowns = shutdowns_all


KeyError: 'ESP discharge pressure'

# 2 - MP raw

In [ ]:
builder = pyfade.DataFrameBuilder(sub_data)
dataset = builder.set_extrapolation(pyfade.Extrapolation.CONSTANT)\
            .set_interpolation(pyfade.Interpolation.LINEAR)\
            .set_filter(pyfade.FilterType.BUTTER_LOW,4,1/12)\
            .build()

w_builder = pyfade.WindowBuilder(dataset)
window = w_builder.set_window_size(SUB_SIZE).build()

group_builder = pyfade.FeatureGroupBuilder(pyfade.FeatureGroups.MatrixProfile, window)
group_builder.set_parameters(MP_PARAMS)
group_builder.build()


fig,axs = dataset.plot(height=2,width=7);
window.plot_feature(pyfade.Features.MatProfileVal,fig=fig,axs=axs);



In [ ]:

# start = SUB_SIZE*12
# builder = pyfade.DataFrameBuilder(sub_data.iloc[:start])
# incremented_dataset = builder.set_extrapolation(pyfade.Extrapolation.CONSTANT)\
#             .set_interpolation(pyfade.Interpolation.LINEAR)\
#             .build()

# w_builder = pyfade.WindowBuilder(incremented_dataset)
# window = w_builder.set_window_size(SUB_SIZE).build()

# group_builder = pyfade.FeatureGroupBuilder(pyfade.FeatureGroups.MatrixProfile, window)
# group_builder.set_parameters(MP_PARAMS)
# group_builder.build()



# end = start + SUB_SIZE*15
# k = 0
# while end < sub_data.size:
#     incremented_dataset.insert_data(sub_data.iloc[start:end])
#     start = end
#     end += SUB_SIZE*7

#     # fig,axs = incremented_dataset.plot(height=3,width=10);
#     # window.plot_feature(pyfade.Features.MatProfileVal,fig=fig,axs=axs,xlimits=[sub_data.index.min(),sub_data.index.max()]);
#     # k+= 1


# incremented_dataset.insert_data(sub_data.iloc[start:])
# fig,axs = incremented_dataset.plot(height=3,width=10);
# window.plot_feature(pyfade.Features.MatProfileVal,fig=fig,axs=axs);

# # fig.savefig(f"./Figures/Plot {k:04d}")



In [ ]:
pyfade.FeatureGroupBuilder(pyfade.FeatureGroups.KProfile,window).build()

window.plot_feature(pyfade.Features.KProfileVal,title='K-Dimensional Profile');

# 3 - Decomposition

In [ ]:
group_builder = pyfade.FeatureGroupBuilder(pyfade.FeatureGroups.WaveletProfile,window)
group_builder.set_parameters(MP_PARAMS).set_parameters(WAVELET_PARAMS)
group_builder.build()

window.get_feature('decomposition')[0].plot();

# 4 - MP wavelet

In [ ]:
group_builder = pyfade.FeatureGroupBuilder(pyfade.FeatureGroups.WaveletProfile,window,replace=True)
group_builder.set_parameters(MP_PARAMS).set_parameters(WAVELET_PARAMS)
group_builder.build()

window.plot_feature(pyfade.Features.WaveletMP,title='WaveletMP');


In [ ]:
group_builder = pyfade.FeatureGroupBuilder(pyfade.FeatureGroups.WaveletKProfile,window,replace=True)
group_builder.build()

window.plot_feature(pyfade.Features.WaveletKP,title='WaveletKP');


In [ ]:
fig.savefig('All_MP.png', format='png', dpi=300)

In [ ]:
KDP = pyfade.get_KDP(sub_data,SUB_SIZE,pre_calc_MP=MPs,shutdowns=shutdowns,**mp_properties)
fig, axs = pyfade.plot_multiple(KDP,same_limits=False,height=1, width=10,linewidth=2,style='r-');

In [ ]:
fig.savefig('All_KDP.png', format='png', dpi=300)